In [4]:
from typing import TypedDict

from langgraph.graph import StateGraph, END

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

import os
from dotenv import load_dotenv

load_dotenv()


llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    api_key=os.getenv("ROUTER_API_TOKEN"),
    base_url="https://openrouter.ai/api/v1",
    temperature=0.5
)

# STATE

class AgentState(TypedDict):
    user_input: str
    plan: str
    code: str
    review: str
# PLANNER AGENT

def planner_agent(state: AgentState):

    prompt = f"""
    Create a step-by-step plan for:
    {state['user_input']}
    """

    response = llm.invoke(prompt)

    return {
        "plan": response.content
    }

# CODER AGENT

def coder_agent(state: AgentState):

    prompt = f"""
    Write Python code for this plan:

    {state['plan']}
    """

    response = llm.invoke(prompt)

    return {
        "code": response.content
    }
# REVIEWER AGENT

def reviewer_agent(state: AgentState):

    prompt = f"""
    Review this code:

    {state['code']}

    Check:
    - bugs
    - improvements
    - optimization
    """

    response = llm.invoke(prompt)

    return {
        "review": response.content
    }
# GRAPH
graph = StateGraph(AgentState)

graph.add_node("planner", planner_agent)
graph.add_node("coder", coder_agent)
graph.add_node("reviewer", reviewer_agent)

graph.set_entry_point("planner")

graph.add_edge("planner", "coder")
graph.add_edge("coder", "reviewer")
graph.add_edge("reviewer", END)

app = graph.compile()

# RUN

result = app.invoke({
    "user_input": "Create a REST API using FastAPI"
})

print(result["plan"])

Creating a REST API using FastAPI can be accomplished in several steps. Below is a step-by-step plan to guide you through the process.

### Step 1: Set Up Your Environment

1. **Install Python**: Ensure you have Python 3.7 or later installed on your machine.
   
2. **Create a Virtual Environment** (optional but recommended):
   ```bash
   python -m venv myenv
   source myenv/bin/activate  # On Windows use `myenv\Scripts\activate`
   ```

3. **Install FastAPI and an ASGI server** (like Uvicorn):
   ```bash
   pip install fastapi uvicorn
   ```

### Step 2: Create a Basic FastAPI Application

1. **Create a new directory for your project**:
   ```bash
   mkdir my_fastapi_app
   cd my_fastapi_app
   ```

2. **Create a new Python file** (e.g., `main.py`):
   ```python
   from fastapi import FastAPI

   app = FastAPI()

   @app.get("/")
   async def read_root():
       return {"Hello": "World"}
   ```

### Step 3: Run Your FastAPI Application

1. **Run the application using Uvicorn**:
   ```